# Análisis Exploratorio de Datos - Campaña de Marketing Bancario

Este notebook realiza un EDA sobre los datos de una campaña de marketing de un banco portugués.
El objetivo es entender mejor los datos y sacar conclusiones útiles sobre los clientes y la campaña.

**Autora:** Ana García  
**Fecha:** Marzo 2024  
**Dataset:** bank-additional.csv + customer-details.xlsx

## 1. Importación de librerías y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 30)

In [ ]:
# Carga del CSV principal
df_bank = pd.read_csv('../datos/raw/bank-additional.csv', index_col=0)
print(f'Dimensiones del dataset: {df_bank.shape}')
df_bank.head()

In [ ]:
# Carga del Excel con datos de clientes (3 hojas: 2012, 2013, 2014)
df_2012 = pd.read_excel('../datos/raw/customer-details.xlsx', sheet_name='2012')
df_2013 = pd.read_excel('../datos/raw/customer-details.xlsx', sheet_name='2013')
df_2014 = pd.read_excel('../datos/raw/customer-details.xlsx', sheet_name='2014')

# Unimos las tres hojas en un solo dataframe
df_customers = pd.concat([df_2012, df_2013, df_2014], ignore_index=True)
print(f'Total de clientes: {df_customers.shape[0]}')
df_customers.head()

## 2. Exploración inicial de los datos

In [ ]:
# Información general del dataframe principal
df_bank.info()

In [ ]:
# Estadísticas descriptivas básicas
df_bank.describe()

In [ ]:
# Revisamos los valores nulos
nulos = df_bank.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0])

In [ ]:
# Porcentaje de nulos
pct_nulos = (df_bank.isnull().sum() / len(df_bank)) * 100
print('Porcentaje de nulos:')
print(pct_nulos[pct_nulos > 0].sort_values(ascending=False))

## 3. Limpieza y Transformación de Datos

In [ ]:
# Hacemos una copia para no modificar el original
df = df_bank.copy()

# --- Columna 'age' ---
# Hay algunos nulos, los rellenamos con la mediana
mediana_age = df['age'].median()
df['age'].fillna(mediana_age, inplace=True)
print(f'Mediana de edad usada para imputación: {mediana_age}')

In [ ]:
# Corregimos los números decimales que usan coma como separador
# IMPORTANTE: esto hay que hacerlo antes de calcular medianas
cols_decimal = ['cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
for col in cols_decimal:
    if df[col].dtype == object:
        df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')

print(df[['cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']].dtypes)

In [ ]:
# --- Columna 'housing' ---
# Tiene algunos nulos, los rellenamos con la moda
moda_housing = df['housing'].mode()[0]
df['housing'].fillna(moda_housing, inplace=True)

print('Nulos restantes:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Estandarizamos la columna 'marital': pasamos todo a minúsculas
df['marital'] = df['marital'].str.lower()
print('Valores únicos en marital:', df['marital'].unique())

In [ ]:
# Transformamos la columna 'date' a formato datetime
# El formato viene como 'día-mes-año' en español
meses = {
    'enero': 'January', 'febrero': 'February', 'marzo': 'March',
    'abril': 'April', 'mayo': 'May', 'junio': 'June',
    'julio': 'July', 'agosto': 'August', 'septiembre': 'September',
    'octubre': 'October', 'noviembre': 'November', 'diciembre': 'December'
}

def convertir_fecha(fecha_str):
    try:
        partes = fecha_str.split('-')
        dia = partes[0]
        mes = meses.get(partes[1].lower(), partes[1])
        anio = partes[2]
        return pd.to_datetime(f'{dia}-{mes}-{anio}', format='%d-%B-%Y')
    except:
        return pd.NaT

df['date'] = df['date'].apply(convertir_fecha)
print(f'Fechas convertidas. Rango: {df["date"].min()} a {df["date"].max()}')

In [ ]:
# Convertimos la variable objetivo 'y' a binario (1/0)
df['y_binary'] = df['y'].map({'yes': 1, 'no': 0})
print('Distribución de la variable objetivo:')
print(df['y'].value_counts())

In [ ]:
# Merge con el dataset de clientes usando el ID
# Renombramos la columna ID para hacer el join
df_customers_clean = df_customers.rename(columns={'ID': 'id_'})

df_merged = df.merge(df_customers_clean, on='id_', how='left')
print(f'Dimensiones tras el merge: {df_merged.shape}')
print(f'Clientes con datos de income: {df_merged["Income"].notna().sum()}')

In [ ]:
# Guardamos el dataset limpio
df_merged.to_csv('../datos/procesados/bank_limpio.csv', index=False)
print('Dataset limpio guardado correctamente.')

## 4. Análisis Descriptivo

In [ ]:
# Estadísticas de las variables numéricas más relevantes
cols_numericas = ['age', 'duration', 'campaign', 'pdays', 'previous', 'Income']
df_merged[cols_numericas].describe().round(2)

In [ ]:
# Análisis por trabajo
print('Distribución por tipo de trabajo:')
print(df_merged['job'].value_counts())
print(f'\nTasa de conversión media: {df_merged["y_binary"].mean():.2%}')

In [ ]:
# Tasa de suscripción por categorías
print('Tasa de suscripción por trabajo:')
tasa_job = df_merged.groupby('job')['y_binary'].mean().sort_values(ascending=False)
print(tasa_job.round(3))

In [ ]:
# Correlación entre variables numéricas
cols_corr = ['age', 'duration', 'campaign', 'previous', 'emp.var.rate', 
             'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y_binary']
# Usamos df_merged pero asegurándonos de que las columnas son numéricas
df_corr = df_merged[cols_corr].copy()
for c in cols_corr:
    df_corr[c] = pd.to_numeric(df_corr[c], errors='coerce')
correlacion = df_corr.corr()
print('Correlaciones con la variable objetivo:')
print(correlacion['y_binary'].sort_values(ascending=False))

## 5. Visualización de Datos

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
df_merged['y'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Distribución de la Variable Objetivo (y)', fontsize=14)
axes[0].set_xlabel('¿Suscribió el depósito?')
axes[0].set_ylabel('Cantidad de clientes')
axes[0].tick_params(rotation=0)

# Gráfico de tarta
df_merged['y'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                    colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Proporción de Suscripciones', fontsize=14)
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../datos/procesados/plot_variable_objetivo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución de edades
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_merged['age'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución de Edades', fontsize=14)
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df_merged['age'].mean(), color='red', linestyle='--', label=f'Media: {df_merged["age"].mean():.1f}')
axes[0].legend()

# Boxplot de edad por suscripción
df_merged.boxplot(column='age', by='y', ax=axes[1], 
                  boxprops=dict(color='steelblue'),
                  medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Edad según Suscripción')
axes[1].set_xlabel('¿Suscribió?')
axes[1].set_ylabel('Edad')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../datos/procesados/plot_edades.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mapa de correlaciones
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(correlacion, dtype=bool))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', 
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlación de Variables Numéricas', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../datos/procesados/plot_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tasa de suscripción por trabajo
tasa_job_df = df_merged.groupby('job')['y_binary'].mean().sort_values(ascending=False).reset_index()
tasa_job_df.columns = ['job', 'tasa_suscripcion']

plt.figure(figsize=(12, 6))
barplot = sns.barplot(data=tasa_job_df, x='job', y='tasa_suscripcion', palette='viridis')
plt.title('Tasa de Suscripción por Tipo de Trabajo', fontsize=14)
plt.xlabel('Tipo de trabajo')
plt.ylabel('Tasa de suscripción')
plt.xticks(rotation=45, ha='right')

# Añadir etiquetas de porcentaje
for bar in barplot.patches:
    barplot.annotate(f'{bar.get_height():.1%}',
                     (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                     ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../datos/procesados/plot_suscripcion_trabajo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Duración de la llamada vs suscripción
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de duración
for label, grupo in df_merged.groupby('y'):
    axes[0].hist(grupo['duration'], bins=50, alpha=0.6, label=label)
axes[0].set_title('Duración de la Llamada por Resultado', fontsize=13)
axes[0].set_xlabel('Duración (segundos)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend(title='Suscribió')
axes[0].set_xlim(0, 2500)

# Media de duración por resultado
media_dur = df_merged.groupby('y')['duration'].mean()
media_dur.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1].set_title('Duración Media por Resultado', fontsize=13)
axes[1].set_xlabel('¿Suscribió?')
axes[1].set_ylabel('Duración media (seg)')
axes[1].tick_params(rotation=0)
for i, v in enumerate(media_dur):
    axes[1].text(i, v + 5, f'{v:.0f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../datos/procesados/plot_duracion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evolución de contactos por mes (contact_month si existe, o extraemos de date)
df_merged['mes'] = df_merged['date'].dt.month
contactos_mes = df_merged.groupby('mes').size().reset_index(name='num_contactos')

plt.figure(figsize=(10, 5))
plt.plot(contactos_mes['mes'], contactos_mes['num_contactos'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(contactos_mes['mes'], contactos_mes['num_contactos'], alpha=0.2, color='steelblue')
plt.title('Número de Contactos por Mes', fontsize=14)
plt.xlabel('Mes')
plt.ylabel('Nº de contactos')
meses_labels = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
                'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
plt.xticks(range(1, 13), meses_labels)
plt.tight_layout()
plt.savefig('../datos/procesados/plot_contactos_mes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis de Income si hay datos suficientes
df_income = df_merged.dropna(subset=['Income'])
print(f'Registros con Income disponible: {len(df_income)}')

if len(df_income) > 100:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df_income['Income'], bins=40, color='mediumpurple', edgecolor='white', alpha=0.8)
    axes[0].set_title('Distribución de Ingresos de Clientes', fontsize=13)
    axes[0].set_xlabel('Income (€)')
    axes[0].set_ylabel('Frecuencia')
    
    sns.boxplot(data=df_income, x='y', y='Income', palette=['#e74c3c', '#2ecc71'], ax=axes[1])
    axes[1].set_title('Ingresos según Suscripción', fontsize=13)
    axes[1].set_xlabel('¿Suscribió?')
    axes[1].set_ylabel('Income (€)')
    
    plt.tight_layout()
    plt.savefig('../datos/procesados/plot_income.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Análisis Adicional y Conclusiones

In [ ]:
# Análisis por educación
tasa_edu = df_merged.groupby('education')['y_binary'].agg(['mean', 'count']).reset_index()
tasa_edu.columns = ['education', 'tasa', 'total']
tasa_edu = tasa_edu.sort_values('tasa', ascending=False)
print('Tasa de suscripción por nivel educativo:')
print(tasa_edu.to_string(index=False))

In [ ]:
# Análisis de la campaña actual vs resultados
# Número de contactos en la campaña
print('Estadísticas del número de contactos por cliente:')
print(df_merged['campaign'].describe())
print(f'\nMáximo de contactos realizados: {df_merged["campaign"].max()}')

# Análisis: ¿más contactos = más suscripciones?
tasa_campaign = df_merged[df_merged['campaign'] <= 10].groupby('campaign')['y_binary'].mean()
print('\nTasa de éxito según nº de contactos (hasta 10):')
print(tasa_campaign.round(3))

In [ ]:
# Agrupación por estado civil y suscripción
tabla_marital = pd.crosstab(df_merged['marital'], df_merged['y'], normalize='index').round(3)
print('Proporción de suscripciones por estado civil:')
print(tabla_marital)

In [ ]:
# Función para calcular métricas de resumen por variable categórica
def resumen_categorica(df, col_cat, col_target='y_binary'):
    """Calcula la tasa de conversión y el tamaño de muestra por categoría."""
    resultado = df.groupby(col_cat)[col_target].agg(
        tasa_conversion='mean',
        n_clientes='count'
    ).sort_values('tasa_conversion', ascending=False)
    resultado['tasa_conversion'] = resultado['tasa_conversion'].round(3)
    return resultado

# Aplicamos la función
print('Resumen por tipo de contacto:')
print(resumen_categorica(df_merged, 'contact'))
print('\nResumen por resultado campaña anterior:')
print(resumen_categorica(df_merged, 'poutcome'))

## 7. Conclusiones del Análisis

A partir del análisis exploratorio realizado, podemos extraer las siguientes conclusiones:

**Variable objetivo**: El dataset está claramente **desbalanceado**, con aproximadamente el 11% de clientes que suscribieron el depósito. Esto es importante tenerlo en cuenta si en el futuro se quiere construir un modelo predictivo.

**Duración de la llamada**: Es la variable que muestra **mayor correlación** con la suscripción. Los clientes que suscribieron tuvieron llamadas significativamente más largas (media ~550 seg vs ~220 seg). Sin embargo, este dato solo se conoce al final de la llamada, por lo que no sería útil como predictor previo.

**Trabajo y educación**: Los estudiantes y jubilados muestran tasas de conversión más altas. El nivel universitario también correlaciona positivamente con la suscripción.

**Campaña anterior**: Los clientes con resultado exitoso en campañas anteriores (poutcome = 'success') tienen una tasa de conversión mucho mayor (~65% vs ~11% general).

**Variables macroeconómicas**: El euribor3m y la tasa de variación del empleo muestran correlación negativa significativa con las suscripciones, lo que tiene sentido económico: cuando los tipos de interés son bajos, los depósitos a plazo son menos atractivos.

**Número de contactos**: Sorprendentemente, más contactos no garantizan mayor éxito. De hecho, la tasa de conversión tiende a bajar después de los primeros 2-3 contactos.

**Ingresos**: Los clientes con datos de income disponible no muestran diferencias muy marcadas en la suscripción, aunque se observa una ligera tendencia positiva en ingresos más altos.